# 환경설정

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd


PROJECT_ID = "your-gcp-project"

client = bigquery.Client(project=PROJECT_ID)


In [ ]:
import pandas as pd
import numpy as np
!pip install koreanize_matplotlib

import koreanize_matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
%%bigquery df_users --project your-gcp-project

SELECT *
FROM `your-gcp-project.stage.stg_users`
LIMIT 10



In [ ]:
df_users

# 전체 사용자 수와 가입 시기별 추이

### 전체 사용자수

In [ ]:
%%bigquery df_total --project your-gcp-project

SELECT  count(distinct user_id) AS TOTAL_USER_COUNT
FROM `your-gcp-project.stage.stg_users`
where is_valid_user = true ;



In [ ]:
df_total

### 가입 시기별 추이

#### 일별

In [ ]:
%%bigquery df_signup_day --project your-gcp-project

SELECT
DATE(signup_at) AS signup_date,
COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users`
GROUP BY signup_date
ORDER BY signup_date

In [ ]:
plt.figure(figsize =(12,5))
plt.plot(df_signup_day['signup_date'], df_signup_day['users'])

plt.title('일별 가입')
plt.xlabel('일자')
plt.ylabel('유저수')

plt.grid(True)
plt.show()

- 2023-05 ~ 2023-06:가입수가 크게 늘고 변동성이 되게 큰 경향을 보임


#### 월별

In [ ]:
%%bigquery df_signup_month --project your-gcp-project

SELECT

DATE_TRUNC(DATE(signup_at),MONTH) AS signup_month,

COUNT(*) AS users

FROM `your-gcp-project.stage.stg_users`

GROUP BY signup_month

ORDER BY signup_month

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(df_signup_month["signup_month"].astype(str),
        df_signup_month["users"])

plt.xticks(rotation=45)

plt.title("Monthly Signups")

plt.show()

#### 누적 가입자

In [ ]:
df_signup_day["cum_users"] = df_signup_day["users"].cumsum()

plt.figure(figsize=(12,5))

plt.plot(df_signup_day["signup_date"], df_signup_day["cum_users"])

plt.title("누적 가입자 수")
plt.xlabel("날짜")
plt.ylabel("누적 사용자")

plt.show()

#### 2023-05만

In [ ]:
import pandas as pd

df_signup_day["signup_date"] = pd.to_datetime(df_signup_day["signup_date"])

may = df_signup_day[
    (df_signup_day["signup_date"] >= "2023-05-01") &
    (df_signup_day["signup_date"] <= "2023-05-31")
]

plt.figure(figsize=(12,5))

plt.plot(may["signup_date"], may["users"], marker="o")

plt.title("2023년 5월 일별 가입자")

plt.xticks(rotation=45)

plt.show()

### 성별

In [ ]:
%%bigquery df_gender --project your-gcp-project

SELECT
gender,
count(*) as users

FROM `your-gcp-project.stage.stg_users`
where gender is not null
GROUP BY gender;


In [ ]:
plt.figure(figsize=(5,5))

plt.bar(df_gender["gender"],
        df_gender["users"])

plt.title("Gender Distribution")

plt.show()

In [ ]:
%%bigquery df_gender_ratio --project your-gcp-project
SELECT
    gender,
    COUNT(*) AS users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS ratio
FROM stage.stg_users
GROUP BY gender;

In [ ]:
df_gender_ratio

- 남성 < 여성 : 여성이 남성보다 가입수? 이용자수? 가 더 많다
- 여성: 58.58
- 남성: 약 41

#### 가입 추이 성별 비교

In [ ]:
%%bigquery df_gender_created --project your-gcp-project

SELECT
DATE_TRUNC(DATE(signup_at),MONTH) month,
gender,
COUNT(*) users
FROM stage.stg_users
GROUP BY month, gender
ORDER BY month;

In [ ]:
df_gender_created

In [ ]:
import matplotlib.pyplot as plt

pivot = df_gender_created.pivot(index='month',
                 columns='gender',
                 values='users')

plt.figure(figsize=(12,5))

plt.plot(pivot.index, pivot["F"], marker='o', label="Female")
plt.plot(pivot.index, pivot["M"], marker='o', label="Male")

plt.title("Monthly User Signups by Gender")
plt.xlabel("Month")
plt.ylabel("Users")
plt.legend()

plt.xticks(rotation=45)

plt.show()

### 학년별

In [ ]:
%%bigquery df_grade --project your-gcp-project

SELECT
    g.grade,
    COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users` AS u
JOIN `your-gcp-project.stage.stg_group` AS g
    ON u.group_id = g.group_id
GROUP BY g.grade
ORDER BY g.grade;

In [ ]:
df_grade

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    df_grade["grade"].astype(str),
    df_grade["users"]
)

plt.title("학년별 사용자 수")
plt.xlabel("학년")
plt.ylabel("사용자 수")

plt.show()

In [ ]:
df_grade["ratio"] = (
    df_grade["users"]
    / df_grade["users"].sum()
    * 100
).round(1)

df_grade

- 2학녀, 3학년, 1학년 순

### 학교별

In [ ]:
%%bigquery df_school --project your-gcp-project

SELECT
    g.school_id,
    COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users` AS u
JOIN `your-gcp-project.stage.stg_group` AS g
    ON u.group_id = g.group_id
GROUP BY g.school_id
ORDER BY users DESC;

In [ ]:
df_school

In [ ]:
%%bigquery df_school_top10_sum --project your-gcp-project

with x as (SELECT
    g.school_id,
    COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users` AS u
JOIN `your-gcp-project.stage.stg_group` AS g
    ON u.group_id = g.group_id
GROUP BY g.school_id
ORDER BY users DESC
limit 20)

select
sum(users)
from x


In [ ]:
df_school_top10_sum

#### 학교 TOP 20

In [ ]:
df_school_top20 = df_school.head(20)

plt.figure(figsize=(12, 6))

plt.bar(
    df_school_top20["school_id"].astype(str),
    df_school_top20["users"]
)

plt.title("사용자 수 상위 20개 학교")
plt.xlabel("학교 ID")
plt.ylabel("사용자 수")
plt.xticks(rotation=45)

plt.show()

#### 학교 x 학년

In [ ]:
%%bigquery df_school_grade --project your-gcp-project

SELECT
    g.school_id,
    g.grade,
    COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users` AS u
JOIN `your-gcp-project.stage.stg_group` AS g
    ON u.group_id = g.group_id
GROUP BY
    g.school_id,
    g.grade
ORDER BY
    users DESC;

In [ ]:
df_school_grade

#### 학년 x 학교 top10

In [ ]:
top10_school_ids = (
    df_school
    .head(10)["school_id"]
    .tolist()
)

df_school_grade_top10 = df_school_grade[
    df_school_grade["school_id"].isin(top10_school_ids)
]

pivot_school_grade = (
    df_school_grade_top10
    .pivot(
        index="school_id",
        columns="grade",
        values="users"
    )
    .fillna(0)
)

pivot_school_grade

In [ ]:
pivot_school_grade.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

plt.title("상위 10개 학교의 학년별 사용자 구성")
plt.xlabel("학교 ID")
plt.ylabel("사용자 수")
plt.xticks(rotation=45)

plt.show()

#### 학교별 사용자 수의 분포

In [ ]:
%%bigquery df_school_distribution --project your-gcp-project

WITH school_users AS (
    SELECT
        g.school_id,
        COUNT(*) AS users
    FROM `your-gcp-project.stage.stg_users` u
    JOIN `your-gcp-project.stage.stg_group` g
        ON u.group_id = g.group_id
    GROUP BY g.school_id
)

SELECT
    MIN(users) AS min_users,
    MAX(users) AS max_users,
    AVG(users) AS avg_users,
    APPROX_QUANTILES(users, 4) AS quartiles
FROM school_users;

In [ ]:
 df_school_distribution

In [ ]:
plt.figure(figsize=(8,2))

plt.boxplot(df_school["users"], vert=False)

plt.title("학교별 사용자 수 Boxplot")

plt.show()

해석
- 최소값 1명
    - 가입자가 1명뿐인 학교도 존재한다.
    - 모든 학교에서 서비스가 동일한 수준으로 사용되는 것은 아니다.
- 최대값 578명
    - 가장 큰 학교는 578명의 가입자를 보유하고 있다.
    - 학교 간 사용자 규모 차이가 매우 크다.
- 평균 122명
    - 학교당 평균 가입자는 약 122명이다.
    - 하지만 평균만 보면 분포를 오해할 수 있다.
- 사분위수
    - 25%의 학교는 가입자가 25명 이하
    - 절반(중앙값)은 108명 이하
    - 75%의 학교는 189명 이하
    - 일부 학교만 300~500명 수준으로 매우 크다.

- 사용자 수는 일부 학교에 집중되어 있으며, 많은 학교는 상대적으로 적은 가입자를 보유하고 있다.

In [ ]:
%%bigquery df_school --project your-gcp-project

SELECT
    g.school_id,
    COUNT(*) AS users
FROM `your-gcp-project.stage.stg_users` u
JOIN `your-gcp-project.stage.stg_group` g
ON u.group_id = g.group_id
GROUP BY g.school_id
ORDER BY users DESC;

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df_school["users"], bins=30)

plt.title("학교별 사용자 수 분포")
plt.xlabel("학교별 사용자 수")
plt.ylabel("학교 수")

plt.show()